# 171Yb 65 3S1 Rydberg lifetime with rydcalc

This notebook records a diagnostic lifetime calculation for
$|r\rangle = |65\,{}^3S_1, F=3/2, m_F=-3/2\rangle$ using the local `rydcalc` submodule.  The decay-rate call follows the `rydcalc/tutorial.ipynb` convention, where an environment is built with `rydcalc.environment(T_K=...)` and the lifetime is `tau = 1 / gamma`.

Important caveat: the explicit `build_dipole_decay_basis` calculation below is a controlled P-state subset, not a complete replacement for `total_decay`.  Its `nu_min = 3.0` cutoff misses low-lying P channels, so the reported lifetime is biased long and should be treated as a lower-bound decay-rate diagnostic, not as the final physical Rydberg lifetime.

In [1]:
from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path

import numpy as np
from scipy import constants as cs

root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

mpl_cache = Path("/tmp/matplotlib-rydcalc")
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

from neutral_yb.external.rydcalc_adapter import (  # noqa: E402
    build_yb171_atom,
    has_arc_c_extension,
    load_rydcalc,
)

rydcalc = load_rydcalc(require_c_extension=True)
from rydcalc.single_basis import state_mqdt  # noqa: E402

yb = build_yb171_atom(cpp_numerov=True, use_db=False, require_c_extension=True)
initial_state = yb.get_state((65, 0, 1.5, -1.5), tt="vlfm")

print(f"project root: {root}")
print(f"rydcalc ARC C extension available: {has_arc_c_extension()}")
print(f"state: {initial_state}")
print(f"v_exact/nub: {initial_state.nub:.8f}")
print(f"energy rel. upper HF limit: {initial_state.get_energy_Hz() / 1e12:.12f} THz")

project root: /home/eris/projects/Noise-Tolerant-Quantum-Control-Optimization
rydcalc ARC C extension available: True
state: |171Yb:64.56,L=0,F=1.5,-1.5>
v_exact/nub: 64.56106363
energy rel. upper HF limit: -0.786121460119 THz


In [2]:
import signal


class DirectTotalDecayTimeout(TimeoutError):
    pass


def _raise_direct_total_decay_timeout(signum, frame):
    raise DirectTotalDecayTimeout("yb.total_decay did not finish before the timeout")


def direct_total_decay_check(atom, state, *, temperature_k: float, timeout_s: int = 60):
    env = rydcalc.environment(T_K=temperature_k)
    old_handler = signal.signal(signal.SIGALRM, _raise_direct_total_decay_timeout)
    signal.alarm(timeout_s)
    try:
        gamma = atom.total_decay(state, env)
        return {
            "temperature_K": temperature_k,
            "gamma_s^-1": gamma,
            "tau_s": 1 / gamma,
            "tau_us": 1e6 / gamma,
            "error": None,
        }
    except Exception as exc:
        return {
            "temperature_K": temperature_k,
            "gamma_s^-1": None,
            "tau_s": None,
            "tau_us": None,
            "error": f"{type(exc).__name__}: {exc}",
        }
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)


direct_total_decay_results = [
    direct_total_decay_check(yb, initial_state, temperature_k=temperature, timeout_s=60)
    for temperature in (0, 300)
]

print("Direct rydcalc total_decay check")
for result in direct_total_decay_results:
    print(f"T = {result['temperature_K']:g} K")
    if result["error"] is None:
        print(f"  gamma = {result['gamma_s^-1']:.6e} s^-1")
        print(f"  tau   = {result['tau_s']:.6e} s = {result['tau_us']:.3f} us")
    else:
        print(f"  yb.total_decay(initial_state, env) failed: {result['error']}")
    print()


def build_dipole_decay_basis(atom, state, *, nu_min: float = 3.0, nu_extra: float = 10.0):
    """Build a diagnostic Yb171 P-state subset; this is not a complete lifetime basis."""
    channels = []
    nu_max = state.nub + nu_extra
    target_l = 1
    f_values = np.arange(max(0.5, state.f - 1), state.f + 1.1, 1.0)

    for f_value in f_values:
        matches = [
            item
            for item in atom.mqdt_models
            if item["L"] == target_l and item["F"] == f_value
        ]
        if not matches:
            continue

        solver = matches[0]["model"]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            nua_list, nub_list = solver.boundstatesinrange([nu_min, nu_max])

        for nua, nub in zip(nua_list, nub_list):
            energy_hz = (
                -solver.RydConst_invcm / nub**2
                + solver.ionizationlimits_invcm[solver.nulimb[0]]
            ) * 100 * cs.c
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                coeffs_i, coeffs_alpha = solver.channelcontributions(nub)

            if not np.all(np.isfinite(np.asarray(coeffs_i, dtype=float))):
                continue

            nuapprox = round(nub * 100) / 100
            for m_value in np.arange(state.m - 1, state.m + 1.1, 1.0):
                if not (-f_value <= m_value <= f_value):
                    continue

                final_state = state_mqdt(
                    atom,
                    (nuapprox, -1, f_value, m_value),
                    coeffs_i,
                    coeffs_alpha,
                    solver.channels,
                    energy_Hz=energy_hz,
                    tt="npfm",
                )
                final_state.pretty_str = (
                    f"|{atom.name}:{nuapprox:.2f},L=1,F={f_value:.1f},{m_value:.1f}>"
                )
                final_state.short_str = f"|{nuapprox:.2f},1,{f_value:.1f},{m_value:.1f}>"
                final_state.nua = nua
                final_state.nub = nub
                final_state.whittaker_wfct = False
                channels.append(final_state)

    return channels


def lifetime_summary(atom, state, final_states, *, temperature_k: float):
    env = rydcalc.environment(T_K=temperature_k)
    rows = []
    gamma_total = 0.0

    for final_state in final_states:
        gamma = atom.partial_decay(state, final_state, env)
        if not np.isfinite(gamma):
            continue
        gamma_total += gamma
        if gamma != 0:
            rows.append(
                {
                    "gamma_s^-1": gamma,
                    "tau_us_contribution": 1e6 / gamma,
                    "delta_GHz": (state.get_energy_Hz() - final_state.get_energy_Hz()) / 1e9,
                    "final_state": final_state.pretty_str,
                }
            )

    rows.sort(key=lambda row: row["gamma_s^-1"], reverse=True)
    return {
        "temperature_K": temperature_k,
        "gamma_s^-1": gamma_total,
        "tau_s": 1 / gamma_total,
        "tau_us": 1e6 / gamma_total,
        "nonzero_channels": len(rows),
        "top_channels": rows[:8],
    }

Direct rydcalc total_decay check
T = 0 K
  yb.total_decay(initial_state, env) failed: DirectTotalDecayTimeout: yb.total_decay did not finish before the timeout

T = 300 K
  yb.total_decay(initial_state, env) failed: DirectTotalDecayTimeout: yb.total_decay did not finish before the timeout



In [3]:
NU_MIN = 3.0
NU_EXTRA = 10.0

final_states = build_dipole_decay_basis(
    yb,
    initial_state,
    nu_min=NU_MIN,
    nu_extra=NU_EXTRA,
)
summaries = [
    lifetime_summary(yb, initial_state, final_states, temperature_k=temperature)
    for temperature in (0, 300)
]

print(f"basis cutoff: nu_min={NU_MIN}, nu_max={initial_state.nub + NU_EXTRA:.8f}")
print(f"candidate subset final states: {len(final_states)}")
for summary in summaries:
    print()
    print(f"T = {summary['temperature_K']:g} K")
    print(f"  gamma = {summary['gamma_s^-1']:.6e} s^-1")
    print(f"  tau   = {summary['tau_s']:.6e} s = {summary['tau_us']:.3f} us")
    print(f"  nonzero channels = {summary['nonzero_channels']}")

basis cutoff: nu_min=3.0, nu_max=74.56106363
candidate subset final states: 1680

T = 0 K
  gamma = 1.554823e+03 s^-1
  tau   = 6.431601e-04 s = 643.160 us
  nonzero channels = 1444

T = 300 K
  gamma = 5.596472e+03 s^-1
  tau   = 1.786840e-04 s = 178.684 us
  nonzero channels = 1680


In [4]:
for summary in summaries:
    print(f"T = {summary['temperature_K']:g} K top channels")
    for index, row in enumerate(summary["top_channels"], start=1):
        print(
            f"  {index}. gamma={row['gamma_s^-1']:.6e} s^-1, "
            f"delta={row['delta_GHz']:.3f} GHz, final={row['final_state']}"
        )
    print()

T = 0 K top channels
  1. gamma=1.958988e+02 s^-1, delta=210042.170 GHz, final=|171Yb:3.95,L=1,F=2.5,-2.5>
  2. gamma=1.813806e+02 s^-1, delta=416305.051 GHz, final=|171Yb:2.81,L=1,F=1.5,-1.5>
  3. gamma=1.209204e+02 s^-1, delta=416305.051 GHz, final=|171Yb:2.81,L=1,F=1.5,-0.5>
  4. gamma=7.835954e+01 s^-1, delta=210042.170 GHz, final=|171Yb:3.95,L=1,F=2.5,-1.5>
  5. gamma=6.306797e+01 s^-1, delta=356236.091 GHz, final=|171Yb:3.04,L=1,F=0.5,-0.5>
  6. gamma=5.757277e+01 s^-1, delta=211597.669 GHz, final=|171Yb:3.94,L=1,F=1.5,-1.5>
  7. gamma=5.345073e+01 s^-1, delta=127089.304 GHz, final=|171Yb:5.07,L=1,F=2.5,-2.5>
  8. gamma=3.838184e+01 s^-1, delta=211597.669 GHz, final=|171Yb:3.94,L=1,F=1.5,-0.5>

T = 300 K top channels
  1. gamma=5.104346e+02 s^-1, delta=-12.448 GHz, final=|171Yb:65.08,L=1,F=2.5,-2.5>
  2. gamma=4.835891e+02 s^-1, delta=11.987 GHz, final=|171Yb:64.08,L=1,F=2.5,-2.5>
  3. gamma=2.568838e+02 s^-1, delta=-16.940 GHz, final=|171Yb:65.27,L=1,F=1.5,-1.5>
  4. gamma=2.182

In [5]:
import json


def _candidate_signature(candidate, *, energy_bin_hz: float = 1e6):
    """Signature for removing overlap between NIST and MQDT candidates."""
    return (
        round(candidate["energy_hz"] / energy_bin_hz),
        round(2 * candidate["F"]),
        round(2 * candidate["m"]),
        candidate["parity"],
    )


def _add_unique_candidate(pool, candidate):
    signature = _candidate_signature(candidate)
    if signature in pool:
        # Prefer NIST low-state energies if an overlap is found.
        if pool[signature]["source"] != "NIST" and candidate["source"] == "NIST":
            pool[signature] = candidate
        return False
    pool[signature] = candidate
    return True


def build_nist_p_candidates(atom, state):
    candidates = []
    nist_path = root / "rydcalc" / "rydcalc" / "Yb171_NIST.txt"
    nist_data = json.loads(nist_path.read_text())[1:]

    for entry in nist_data:
        if entry.get("l") != 1:
            continue
        n, spin, ell, j = entry["n"], entry["S"], entry["l"], entry["J"]
        f_values = np.arange(abs(j - atom.I), j + atom.I + 0.1, 1.0)
        for f_value in f_values:
            for m_value in (state.m - 1, state.m, state.m + 1):
                if not (-f_value <= m_value <= f_value):
                    continue
                final_state = atom.get_state((n, spin, ell, j, f_value, m_value), tt="NIST")
                if final_state is None:
                    continue
                candidates.append(
                    {
                        "state": final_state,
                        "source": "NIST",
                        "label": f"NIST {n} {2 * spin + 1:.0f}P{j:g} F={f_value:g} m={m_value:g}",
                        "energy_hz": final_state.get_energy_Hz(),
                        "F": f_value,
                        "m": m_value,
                        "parity": final_state.parity,
                    }
                )
    return candidates


def build_mqdt_high_p_candidates(atom, state, *, nu_min: float = 12.0, nu_extra: float = 10.0):
    candidates = []
    skipped_nonfinite = 0
    nu_max = state.nub + nu_extra

    for f_value in (0.5, 1.5, 2.5):
        matches = [item for item in atom.mqdt_models if item["L"] == 1 and item["F"] == f_value]
        if not matches:
            continue
        solver = matches[0]["model"]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            nua_list, nub_list = solver.boundstatesinrange([nu_min, nu_max])

        for nua, nub in zip(nua_list, nub_list):
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                coeffs_i, coeffs_alpha = solver.channelcontributions(nub)
            if not np.all(np.isfinite(np.asarray(coeffs_i, dtype=float))):
                skipped_nonfinite += 1
                continue

            energy_hz = (
                -solver.RydConst_invcm / nub**2
                + solver.ionizationlimits_invcm[solver.nulimb[0]]
            ) * 100 * cs.c
            nuapprox = round(nub * 100) / 100
            for m_value in (state.m - 1, state.m, state.m + 1):
                if not (-f_value <= m_value <= f_value):
                    continue
                final_state = state_mqdt(
                    atom,
                    (nuapprox, -1, f_value, m_value),
                    coeffs_i,
                    coeffs_alpha,
                    solver.channels,
                    energy_Hz=energy_hz,
                    tt="npfm",
                )
                final_state.pretty_str = f"|{atom.name}:{nuapprox:.2f},L=1,F={f_value:.1f},{m_value:.1f}>"
                final_state.nua = nua
                final_state.nub = nub
                final_state.whittaker_wfct = False
                candidates.append(
                    {
                        "state": final_state,
                        "source": "MQDT",
                        "label": f"MQDT nu={nub:.6g} P F={f_value:g} m={m_value:g}",
                        "energy_hz": final_state.get_energy_Hz(),
                        "F": f_value,
                        "m": m_value,
                        "parity": final_state.parity,
                    }
                )
    return candidates, skipped_nonfinite


def build_importance_candidate_pool(atom, state, *, mqdt_nu_min: float = 12.0):
    nist_candidates = build_nist_p_candidates(atom, state)
    mqdt_candidates, skipped_nonfinite = build_mqdt_high_p_candidates(
        atom,
        state,
        nu_min=mqdt_nu_min,
    )

    unique = {}
    duplicate_count = 0
    for candidate in [*nist_candidates, *mqdt_candidates]:
        if not _add_unique_candidate(unique, candidate):
            duplicate_count += 1

    return {
        "candidates": list(unique.values()),
        "n_nist": len(nist_candidates),
        "n_mqdt": len(mqdt_candidates),
        "n_unique": len(unique),
        "duplicates_removed": duplicate_count,
        "mqdt_skipped_nonfinite": skipped_nonfinite,
    }


def importance_ranked_decay(atom, state, candidates, *, temperature_k: float, coverage: float = 0.95):
    env = rydcalc.environment(T_K=temperature_k)
    rows = []
    skipped_partial_decay = 0

    for candidate in candidates:
        try:
            gamma = atom.partial_decay(state, candidate["state"], env)
        except Exception:
            skipped_partial_decay += 1
            continue
        if not np.isfinite(gamma):
            skipped_partial_decay += 1
            continue
        if gamma <= 0:
            continue
        rows.append(
            {
                "source": candidate["source"],
                "label": candidate["label"],
                "gamma_s^-1": float(gamma),
                "delta_GHz": float((state.get_energy_Hz() - candidate["energy_hz"]) / 1e9),
            }
        )

    rows.sort(key=lambda row: row["gamma_s^-1"], reverse=True)
    total_gamma = sum(row["gamma_s^-1"] for row in rows)
    cumulative = 0.0
    selected = []
    for row in rows:
        cumulative += row["gamma_s^-1"]
        selected.append(row)
        if total_gamma > 0 and cumulative / total_gamma >= coverage:
            break

    return {
        "temperature_K": temperature_k,
        "all_rows": rows,
        "selected_rows": selected,
        "total_gamma_s^-1": total_gamma,
        "total_tau_us": 1e6 / total_gamma,
        "selected_gamma_s^-1": cumulative,
        "selected_fraction": cumulative / total_gamma,
        "skipped_partial_decay": skipped_partial_decay,
    }


MQDT_HIGH_NU_MIN = 12.0
COVERAGE_TARGET = 0.95

importance_pool = build_importance_candidate_pool(yb, initial_state, mqdt_nu_min=MQDT_HIGH_NU_MIN)
importance_results = [
    importance_ranked_decay(
        yb,
        initial_state,
        importance_pool["candidates"],
        temperature_k=temperature,
        coverage=COVERAGE_TARGET,
    )
    for temperature in (0, 300)
]

print("Importance-truncated candidate pool")
print(f"  NIST low-P candidates: {importance_pool['n_nist']}")
print(f"  MQDT high-P candidates, nu >= {MQDT_HIGH_NU_MIN:g}: {importance_pool['n_mqdt']}")
print(f"  unique candidates after energy/F/m de-duplication: {importance_pool['n_unique']}")
print(f"  duplicates removed: {importance_pool['duplicates_removed']}")
print(f"  MQDT candidates skipped for nonfinite coefficients: {importance_pool['mqdt_skipped_nonfinite']}")

for result in importance_results:
    selected = result["selected_rows"]
    selected_by_source = {
        source: sum(row["gamma_s^-1"] for row in selected if row["source"] == source)
        for source in ("NIST", "MQDT")
    }
    print()
    print(f"T = {result['temperature_K']:g} K")
    print(f"  nonzero candidates = {len(result['all_rows'])}")
    print(f"  full candidate gamma = {result['total_gamma_s^-1']:.6e} s^-1")
    print(f"  full candidate tau   = {result['total_tau_us']:.3f} us")
    print(f"  channels for {COVERAGE_TARGET:.0%} coverage = {len(selected)}")
    print(f"  selected gamma = {result['selected_gamma_s^-1']:.6e} s^-1")
    print(f"  selected fraction = {result['selected_fraction']:.6f}")
    print(f"  selected gamma by source = {selected_by_source}")
    print(f"  partial-decay failures skipped = {result['skipped_partial_decay']}")
    print("  top 20 channels:")
    for index, row in enumerate(result["all_rows"][:20], start=1):
        print(
            f"    {index:2d}. gamma={row['gamma_s^-1']:.6e} s^-1, "
            f"delta={row['delta_GHz']:.3f} GHz, {row['source']}, {row['label']}"
        )


Importance-truncated candidate pool
  NIST low-P candidates: 84
  MQDT high-P candidates, nu >= 12: 1458
  unique candidates after energy/F/m de-duplication: 1542
  duplicates removed: 0
  MQDT candidates skipped for nonfinite coefficients: 0

T = 0 K
  nonzero candidates = 1306
  full candidate gamma = 3.262577e+03 s^-1
  full candidate tau   = 306.506 us
  channels for 95% coverage = 83
  selected gamma = 3.099765e+03 s^-1
  selected fraction = 0.950097
  selected gamma by source = {'NIST': 3037.1311470127034, 'MQDT': 62.63425960157512}
  partial-decay failures skipped = 0
  top 20 channels:
     1. gamma=5.444279e+02 s^-1, delta=920555.090 GHz, NIST, NIST 6 3P2 F=2.5 m=-2.5
     2. gamma=2.778149e+02 s^-1, delta=972073.534 GHz, NIST, NIST 6 3P1 F=1.5 m=-1.5
     3. gamma=2.177712e+02 s^-1, delta=920555.090 GHz, NIST, NIST 6 3P2 F=2.5 m=-1.5
     4. gamma=1.872883e+02 s^-1, delta=355702.548 GHz, NIST, NIST 7 3P2 F=2.5 m=-2.5
     5. gamma=1.852241e+02 s^-1, delta=993165.972 GHz, NIST